# BUAN381 Final Project — AI Adoption & Revenue Growth Prediction Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Data Loading

In [ ]:
df = pd.read_csv('../dataset/ai_company_adoption.csv')
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
df.head(3)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['revenue_growth_percent'], bins=50, edgecolor='black', color='steelblue')
axes[0].set_title('Revenue Growth % Distribution')
axes[0].set_xlabel('Revenue Growth (%)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['revenue_growth_percent'])
axes[1].set_title('Revenue Growth % Boxplot')
axes[1].set_ylabel('Revenue Growth (%)')

plt.tight_layout()
plt.savefig('../report/fig_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(df['revenue_growth_percent'].describe().round(2))

In [ ]:
industry_avg = df.groupby('industry')['revenue_growth_percent'].agg(['mean','std','count']).reset_index()
industry_avg.columns = ['industry','mean_revenue_growth','std_revenue_growth','count']
industry_avg = industry_avg.sort_values('mean_revenue_growth', ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(industry_avg['industry'], industry_avg['mean_revenue_growth'], color='steelblue', edgecolor='black')
plt.title('Average Revenue Growth by Industry', fontsize=14, fontweight='bold')
plt.xlabel('Industry')
plt.ylabel('Mean Revenue Growth (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../report/fig_revenue_by_industry.png', dpi=150, bbox_inches='tight')
plt.show()

industry_avg.to_csv('../dataset/tableau_industry_summary.csv', index=False)

In [ ]:
num_cols = ['ai_adoption_rate','ai_maturity_score','ai_budget_percentage','ai_training_hours',
            'num_ai_tools_used','ai_projects_active','task_automation_rate','time_saved_per_week',
            'ai_investment_per_employee','num_employees','annual_revenue_usd_millions',
            'company_age','revenue_growth_percent']

corr = df[num_cols].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../report/fig_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
size_order = ['Startup', 'SME', 'Enterprise']
size_data = [df[df['company_size'] == s]['revenue_growth_percent'].values for s in size_order]

plt.figure(figsize=(10, 6))
plt.boxplot(size_data, tick_labels=size_order)
plt.title('Revenue Growth by Company Size', fontsize=14, fontweight='bold')
plt.xlabel('Company Size')
plt.ylabel('Revenue Growth (%)')
plt.tight_layout()
plt.savefig('../report/fig_revenue_by_size.png', dpi=150, bbox_inches='tight')
plt.show()

for size in size_order:
    mean = df[df['company_size'] == size]['revenue_growth_percent'].mean()
    print(f"{size}: {mean:.2f}%")

In [ ]:
# For Tableau scatter: AI budget vs revenue growth (sample 5000 rows for performance)
tableau_scatter = df[['ai_budget_percentage','revenue_growth_percent','industry',
                       'company_size','region','ai_adoption_stage']].sample(5000, random_state=42)
tableau_scatter.to_csv('../dataset/tableau_scatter.csv', index=False)

# For Tableau map: region averages
region_avg = df.groupby('region').agg(
    mean_revenue_growth=('revenue_growth_percent','mean'),
    mean_ai_maturity=('ai_maturity_score','mean'),
    mean_ai_budget=('ai_budget_percentage','mean'),
    count=('revenue_growth_percent','count')
).reset_index()
region_avg.to_csv('../dataset/tableau_region_summary.csv', index=False)
print("Tableau exports done")

In [ ]:
TARGET = 'revenue_growth_percent'
FEATURES = [c for c in df_model.columns if c != TARGET]

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
DROP_COLS = ['response_id','company_id','survey_source','data_collection_method',
             'country','quarter']

df_model = df.drop(columns=DROP_COLS).copy()

CAT_COLS = ['region','industry','company_size','company_age_group','ai_adoption_stage',
            'ai_primary_tool','ai_use_case','data_privacy_level','ai_ethics_committee']

le = LabelEncoder()
for col in CAT_COLS:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Shape after encoding:", df_model.shape)
print("Dtypes:\n", df_model.dtypes.value_counts())

## 3. Feature Engineering